<a href="https://colab.research.google.com/github/andresanchez256/Portafolio/blob/main/Proyecto_Clasificacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 Análisis de Riesgo Crediticio con Machine Learning

## 📋 Contexto del Problema

El riesgo crediticio es una de las áreas más críticas en la banca y las finanzas. Las instituciones financieras necesitan evaluar la probabilidad de que un solicitante de crédito no pague su préstamo (default) para tomar decisiones informadas.

### 🎯 Objetivo del Proyecto
Desarrollar un modelo de machine learning que prediga la **probabilidad de impago** de un cliente basado en sus características financieras y demográficas, utilizando el **German Credit Dataset**.

### 💰 Impacto en el Negocio
- **Reducción de pérdidas**: Identificar clientes de alto riesgo antes de otorgar créditos
- **Optimización de tasas**: Ajustar tasas de interés según el nivel de riesgo
- **Mejor toma de decisiones**: Complementar el juicio humano con predicciones basadas en datos

### 📊 Métricas de Éxito
- **AUC-ROC**: Capacidad del modelo para distinguir entre buenos y malos pagadores
- **Costo total**: Minimizar el costo de errores (falsos positivos y falsos negativos)
- **Interpretabilidad**: Entender qué variables influyen más en el riesgo

### 📚 Dataset Utilizado
- **Fuente**: UCI Machine Learning Repository - German Credit Dataset
- **Registros**: 1000 solicitudes de crédito
- **Variables**: 24 características (demográficas, financieras, historial crediticio)
- **Clases**: 700 "Buenos pagadores" (70%) y 300 "Malos pagadores" (30%)

In [ ]:
# ============================================
# CONFIGURACIÓN INICIAL DEL NOTEBOOK
# ============================================

# Instalar librerías necesarias (solo si no están instaladas)
!pip install -q pandas numpy scikit-learn matplotlib seaborn

# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.metrics import precision_recall_curve, average_precision_score
import warnings
warnings.filterwarnings('ignore')

# Configurar visualizaciones
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("✅ Librerías importadas correctamente")
print(f"📊 Pandas versión: {pd.__version__}")
print(f"🔢 NumPy versión: {np.__version__}")

✅ Librerías importadas correctamente
📊 Pandas versión: 2.2.2
🔢 NumPy versión: 2.0.2


## 📊 Exploración del Dataset

### Descripción de Variables

El German Credit Dataset contiene 24 variables que describen al solicitante del crédito:

#### Variables Demográficas
- **age**: Edad del solicitante en años
- **personal_status_sex**: Estado civil y género
- **foreign_worker**: Indicador de trabajador extranjero

#### Variables Financieras
- **amount**: Monto solicitado del préstamo (DM)
- **duration_months**: Duración del préstamo en meses
- **installment_rate**: Porcentaje del ingreso destinado a la cuota
- **status_account**: Estado de la cuenta corriente

#### Historial Crediticio
- **credit_history**: Historial de créditos anteriores
- **existing_credits_count**: Número de créditos existentes
- **savings_account**: Estado de la cuenta de ahorros

#### Variables Laborales
- **employment_duration**: Antigüedad en el empleo actual
- **job**: Tipo de ocupación

### Variable Objetivo
- **target**: 0 = Buen pagador, 1 = Mal pagador (impago)

### 🔍 Estructura del Problema
- **Tipo**: Clasificación binaria
- **Desbalanceo**: 70% buenos, 30% malos
- **Contexto**: Matriz de costos (FN = 5, FP = 1)

In [ ]:
# ============================================
# CARGA DE DATOS
# ============================================

# URL del dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data-numeric'

# Definir nombres de variables (23 features)
feature_names = [
    'status_account',      # Estado de cuenta corriente
    'duration_months',     # Duración del préstamo (meses)
    'credit_history',      # Historial crediticio
    'purpose',             # Propósito del préstamo
    'amount',              # Monto solicitado (DM)
    'savings_account',     # Cuenta de ahorros
    'employment_duration', # Antigüedad laboral
    'installment_rate',    # Tasa de cuota (% ingreso)
    'personal_status_sex', # Estado civil y género
    'other_debtors',       # Otros deudores
    'residence_since',     # Años en residencia
    'property',            # Propiedad
    'age',                 # Edad (años)
    'other_installment_plans', # Otros planes de cuota
    'housing',             # Situación de vivienda
    'existing_credits_count', # Número de créditos existentes
    'job',                 # Tipo de trabajo
    'people_liable',       # Personas a cargo
    'telephone',           # Teléfono
    'foreign_worker',      # Trabajador extranjero
    'feature_21',          # Variable adicional
    'feature_22',          # Variable adicional
    'feature_23'           # Variable adicional
]

# Cargar los datos
df = pd.read_csv(url, sep='\s+', header=None)

# Asignar nombres de columnas (23 features + 1 clase)
df.columns = feature_names + ['class']

# Crear variable target (0 = buen pagador, 1 = mal pagador)
df['target'] = df['class'].apply(lambda x: 0 if x == 1 else 1)

# Eliminar columna 'class' original
df = df.drop('class', axis=1)

# Mostrar información básica
print(f"✅ Datos cargados exitosamente")
print(f"📊 Dimensiones: {df.shape[0]} registros, {df.shape[1]} columnas")
print(f"\n📋 Distribución de clases:")
print(f"  - Buenos pagadores (0): {sum(df['target']==0)} ({sum(df['target']==0)/len(df)*100:.1f}%)")
print(f"  - Malos pagadores (1): {sum(df['target']==1)} ({sum(df['target']==1)/len(df)*100:.1f}%)")

## 🔍 Análisis Exploratorio de Datos (EDA)

### Insights Iniciales

1. **Balance de Clases**: El dataset está desbalanceado (70% buenos, 30% malos), lo cual es realista y requiere técnicas especiales.

2. **Variables Numéricas vs Categóricas**: Algunas variables están codificadas numéricamente pero son categóricas (ej. `status_account`). Esto afecta cómo debemos procesarlas.

3. **Escalas Diferentes**: Variables como `amount` y `duration_months` tienen escalas muy diferentes, lo que requiere normalización para modelos como regresión logística.

### Preguntas de Investigación
- ¿Qué características están más correlacionadas con el impago?
- ¿Cómo se distribuyen las variables clave entre buenos y malos pagadores?
- ¿Podemos identificar patrones visuales que distingan a los clientes de alto riesgo?

In [ ]:
# ============================================
# ESTADÍSTICAS DESCRIPTIVAS
# ============================================

# Información general del dataset
print("="*60)
print("INFORMACIÓN GENERAL DEL DATASET")
print("="*60)
print("\n📋 Tipos de datos:")
print(df.dtypes.value_counts())

print("\n📊 Estadísticas descriptivas:")
display(df.describe().round(2))

print("\n🔍 Verificar valores nulos:")
print(df.isnull().sum())

print("\n📋 Primeras 5 filas:")
display(df.head())

print("\n📋 Últimas 5 filas:")
display(df.tail())

## 🎯 Análisis de la Variable Objetivo

### Distribución de Clases

El desbalanceo de clases es un aspecto crítico en problemas de riesgo crediticio:

- **70% Buenos pagadores**: Clase mayoritaria
- **30% Malos pagadores**: Clase minoritaria (la que queremos predecir)

### Implicaciones para el Modelado

1. **Métricas**: Accuracy no es apropiada. Usaremos AUC-ROC y F1-score.
2. **Umbral de Decisión**: Podemos ajustar el umbral para balancear errores.
3. **Matriz de Costos**: Es más costoso no detectar a un mal pagador que rechazar a uno bueno.

### Visualización de Distribución

In [ ]:
# ============================================
# VISUALIZACIÓN DE LA VARIABLE OBJETIVO
# ============================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
class_counts = df['target'].value_counts()
colors = ['#2ecc71', '#e74c3c']
ax1.bar(['Bueno (0)', 'Malo (1)'], class_counts, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_title('Distribución de Clases', fontsize=14, fontweight='bold')
ax1.set_xlabel('Clase')
ax1.set_ylabel('Número de Registros')
for i, v in enumerate(class_counts):
    ax1.text(i, v + 5, str(v), ha='center', va='bottom', fontweight='bold')

# Gráfico de pastel
ax2.pie(class_counts, labels=['Bueno (0)', 'Malo (1)'], autopct='%1.1f%%',
        colors=colors, startangle=90, explode=(0, 0.05), shadow=True)
ax2.set_title('Proporción de Clases', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Resumen estadístico
print("\n📊 Resumen de la variable objetivo:")
print(f"  - Total de registros: {len(df)}")
print(f"  - Buenos pagadores: {sum(df['target']==0)} ({sum(df['target']==0)/len(df)*100:.1f}%)")
print(f"  - Malos pagadores: {sum(df['target']==1)} ({sum(df['target']==1)/len(df)*100:.1f}%)")
print(f"  - Ratio de desbalanceo: {sum(df['target']==0)/sum(df['target']==1):.2f}:1")

## 📈 Análisis de Variables Numéricas Clave

### Variables Seleccionadas para Análisis

1. **amount**: Monto del préstamo - ¿Los préstamos más grandes tienen mayor riesgo?
2. **duration_months**: Duración del préstamo - ¿Plazos más largos implican más riesgo?
3. **age**: Edad del solicitante - ¿La edad influye en la probabilidad de impago?
4. **installment_rate**: Porcentaje del ingreso destinado a la cuota - ¿Mayor tasa = mayor riesgo?

### Hipótesis Iniciales
- **Amount**: Posible correlación positiva con impago (más dinero = más riesgo)
- **Duration**: Posible correlación positiva (más tiempo = más incertidumbre)
- **Age**: Posible correlación negativa (mayor edad = más estabilidad)
- **Installment_rate**: Posible correlación positiva (más esfuerzo = más riesgo)

In [ ]:
# ============================================
# VISUALIZACIÓN DE VARIABLES NUMÉRICAS
# ============================================

# Seleccionar variables numéricas clave
numeric_vars = ['amount', 'duration_months', 'age', 'installment_rate']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, var in enumerate(numeric_vars):
    # Boxplot por clase
    data = [df[df['target']==0][var], df[df['target']==1][var]]
    bp = axes[idx].boxplot(data, labels=['Bueno', 'Malo'], patch_artist=True)

    # Colores
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')

    axes[idx].set_title(f'Distribución de {var} por Clase', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(var)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Estadísticas por clase
print("\n📊 Estadísticas por clase:")
for var in numeric_vars:
    print(f"\n{var}:")
    print(f"  - Buenos pagadores: media={df[df['target']==0][var].mean():.2f}, "
          f"mediana={df[df['target']==0][var].median():.2f}")
    print(f"  - Malos pagadores: media={df[df['target']==1][var].mean():.2f}, "
          f"mediana={df[df['target']==1][var].median():.2f}")

## 🔗 Análisis de Correlaciones

### Matriz de Correlación

La matriz de correlación nos ayuda a identificar:
1. **Relaciones entre variables**: Variables altamente correlacionadas pueden causar multicolinealidad
2. **Relación con el target**: Variables más correlacionadas con el impago son más importantes

### Insights Esperados
- **Correlación positiva**: Variables que aumentan con el riesgo de impago
- **Correlación negativa**: Variables que disminuyen con el riesgo de impago
- **Variables de control**: Variables sin correlación significativa con el target

In [ ]:
# ============================================
# MATRIZ DE CORRELACIÓN
# ============================================

# Calcular matriz de correlación
corr_matrix = df.corr()

# Crear máscara para triángulo superior
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Configurar figura
plt.figure(figsize=(16, 12))

# Heatmap
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8})

plt.title('Matriz de Correlación del German Credit Dataset', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Mostrar correlaciones más fuertes con el target
print("\n📊 Top 10 variables más correlacionadas con el target:")
target_corr = corr_matrix['target'].sort_values(ascending=False)
for var in target_corr.index[1:11]:  # Excluir el target mismo
    print(f"  - {var}: {target_corr[var]:.3f}")

## 🏷️ Análisis de Variables Categóricas

### Identificación de Patrones

Aunque las variables están codificadas numéricamente, muchas son categóricas:
- **status_account**: Estado de cuenta (4 categorías)
- **credit_history**: Historial crediticio (5 categorías)
- **purpose**: Propósito del préstamo (10 categorías)
- **employment_duration**: Antigüedad laboral (5 categorías)

### Análisis por Categoría
Vamos a explorar cómo varía la tasa de impago entre las diferentes categorías de estas variables. Esto nos ayudará a:
1. Identificar segmentos de alto riesgo
2. Entender el comportamiento del cliente
3. Generar hipótesis para el modelado

In [ ]:
# ============================================
# ANÁLISIS DE VARIABLES CATEGÓRICAS
# ============================================

# Seleccionar variables categóricas clave
categorical_vars = ['status_account', 'credit_history', 'purpose', 'employment_duration']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, var in enumerate(categorical_vars):
    # Calcular tasa de impago por categoría
    default_rate = df.groupby(var)['target'].mean().sort_values()

    # Crear barplot
    colors = ['#e74c3c' if x > 0.3 else '#2ecc71' for x in default_rate.values]
    axes[idx].barh(range(len(default_rate)), default_rate.values, color=colors)
    axes[idx].set_yticks(range(len(default_rate)))
    axes[idx].set_yticklabels(default_rate.index)
    axes[idx].set_xlabel('Tasa de Impago')
    axes[idx].set_title(f'Tasa de Impago por {var}', fontsize=12, fontweight='bold')
    axes[idx].axvline(x=0.3, color='red', linestyle='--', alpha=0.5, label='Umbral 30%')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Resumen de categorías de alto riesgo
print("\n🔴 Categorías con mayor tasa de impago (>30%):")
high_risk_vars = []
for var in categorical_vars:
    high_risk = df.groupby(var)['target'].mean()
    high_risk = high_risk[high_risk > 0.3]
    if len(high_risk) > 0:
        print(f"\n{var}:")
        for cat, rate in high_risk.items():
            print(f"  - Categoría {cat}: {rate:.1%}")
            high_risk_vars.append((var, cat, rate))

## 🤖 Preparación para el Modelado

### Procesamiento de Datos

Para preparar los datos para el modelado, realizamos:

1. **Separación de Features y Target**: X (variables predictoras), y (variable objetivo)
2. **División Train/Test**: 70% entrenamiento, 30% prueba (estratificado)
3. **Escalado de Variables**: StandardScaler para normalizar las variables numéricas

### Estrategia de Modelado

Probaremos dos enfoques:

1. **Regresión Logística**: Modelo interpretable, bueno para explicar decisiones
2. **Random Forest**: Modelo más complejo, captura relaciones no lineales

### Métricas de Evaluación

- **AUC-ROC**: Capacidad discriminativa del modelo
- **Matriz de Confusión**: Errores tipo I y II
- **Costo Total**: Incorporando la matriz de costos del negocio

In [ ]:
# ============================================
# PREPARACIÓN DE DATOS PARA MODELADO
# ============================================

# Separar features y target
X = df.drop('target', axis=1)
y = df['target']

print(f"📊 Features (X): {X.shape[0]} registros, {X.shape[1]} variables")
print(f"🎯 Target (y): {len(y)} registros")

# División train/test (estratificado)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\n📊 División de datos:")
print(f"  - Entrenamiento: {len(X_train)} muestras ({len(X_train)/len(X)*100:.1f}%)")
print(f"  - Prueba: {len(X_test)} muestras ({len(X_test)/len(X)*100:.1f}%)")

# Escalado de datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Datos escalados correctamente")
print(f"  - Media (train): {X_train_scaled.mean():.3f}")
print(f"  - Desv. estándar (train): {X_train_scaled.std():.3f}")

## 🚀 Entrenamiento de Modelos

### Modelos Implementados

#### 1. Regresión Logística
- **Ventajas**: Interpretable, rápido, bueno para datos lineales
- **Parámetros**: max_iter=1000, random_state=42
- **Salida**: Probabilidad de impago

#### 2. Random Forest
- **Ventajas**: Captura relaciones no lineales, maneja interacciones
- **Parámetros**: n_estimators=100, random_state=42
- **Salida**: Probabilidad de impago

### Selección del Mejor Modelo
Compararemos ambos modelos usando AUC-ROC y elegiremos el que tenga mejor rendimiento en el conjunto de prueba.

In [ ]:
# ============================================
# ENTRENAMIENTO DE MODELOS
# ============================================

print("🚀 Entrenando modelos...")

# 1. Regresión Logística
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
auc_lr = roc_auc_score(y_test, y_proba_lr)
print(f"\n🤖 Regresión Logística:")
print(f"  - AUC-ROC: {auc_lr:.4f}")

# 2. Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
y_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]
auc_rf = roc_auc_score(y_test, y_proba_rf)
print(f"\n🌲 Random Forest:")
print(f"  - AUC-ROC: {auc_rf:.4f}")

# Seleccionar el mejor modelo
if auc_lr >= auc_rf:
    best_model = lr_model
    best_proba = y_proba_lr
    best_auc = auc_lr
    model_name = "Regresión Logística"
else:
    best_model = rf_model
    best_proba = y_proba_rf
    best_auc = auc_rf
    model_name = "Random Forest"

print(f"\n🏆 Mejor modelo: {model_name}")
print(f"   AUC-ROC: {best_auc:.4f}")

## 📈 Curva ROC

### Interpretación de la Curva ROC

- **Eje X**: Tasa de Falsos Positivos (1 - Especificidad)
- **Eje Y**: Tasa de Verdaderos Positivos (Sensibilidad/Recall)
- **AUC (Área Bajo la Curva)**: Mide la capacidad discriminativa del modelo

### Puntos Clave

- **AUC = 1.0**: Clasificador perfecto
- **AUC = 0.5**: Clasificador aleatorio (sin poder predictivo)
- **AUC > 0.7**: Aceptable para problemas de riesgo crediticio

### Comparación de Modelos

In [ ]:
# ============================================
# CURVA ROC - COMPARACIÓN DE MODELOS
# ============================================

plt.figure(figsize=(10, 8))

# Curva ROC para Regresión Logística
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
plt.plot(fpr_lr, tpr_lr, label=f'Regresión Logística (AUC = {auc_lr:.4f})',
         linewidth=2, color='blue')

# Curva ROC para Random Forest
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.4f})',
         linewidth=2, color='green')

# Línea de referencia (modelo aleatorio)
plt.plot([0, 1], [0, 1], 'k--', label='Modelo Aleatorio (AUC = 0.5)',
         linewidth=1, alpha=0.5)

plt.xlabel('Tasa de Falsos Positivos (1 - Especificidad)', fontsize=12)
plt.ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontsize=12)
plt.title('Curva ROC - Comparación de Modelos', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Resumen de AUC
print(f"\n📊 Resumen de AUC:")
print(f"  - Regresión Logística: {auc_lr:.4f}")
print(f"  - Random Forest: {auc_rf:.4f}")
print(f"  - Mejor modelo: {model_name} (AUC = {best_auc:.4f})")

## 💰 Optimización con Matriz de Costos

### Matriz de Costos del Negocio

| Real \\ Predicho | Bueno | Malo |
|------------------|-------|------|
| **Bueno**        | 0     | 1    |
| **Malo**         | 5     | 0    |

### Interpretación
- **Falso Negativo (FN = 5)**: Clasificar un mal cliente como bueno. **MUY COSTOSO** (el banco pierde dinero)
- **Falso Positivo (FP = 1)**: Clasificar un buen cliente como malo. **POCO COSTOSO** (el banco pierde una oportunidad)

### Objetivo
Encontrar el umbral de probabilidad que minimice el **costo total**:
- **Costo Total = (FN × 5) + (FP × 1)**

### Implicaciones
Un umbral más bajo (ej. 0.3) clasificará a más clientes como "malos", reduciendo FN pero aumentando FP.
Un umbral más alto (ej. 0.7) clasificará a más clientes como "buenos", reduciendo FP pero aumentando FN.

In [ ]:
# ============================================
# OPTIMIZACIÓN DE UMBRAL CON MATRIZ DE COSTOS
# ============================================

# Función para calcular costo total
def calcular_costo(y_true, y_proba, umbral, cost_FN=5, cost_FP=1):
    y_pred = (y_proba >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    costo_total = (fn * cost_FN) + (fp * cost_FP)
    return costo_total, tn, fp, fn, tp

# Probar diferentes umbrales
umbrales = np.arange(0.1, 0.9, 0.05)
costos = []
resultados = []

print("🔄 Probando diferentes umbrales:")
for umbral in umbrales:
    costo, tn, fp, fn, tp = calcular_costo(y_test, best_proba, umbral)
    costos.append(costo)
    resultados.append({
        'umbral': umbral,
        'costo': costo,
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
        'Precision': tp/(tp+fp) if (tp+fp)>0 else 0,
        'Recall': tp/(tp+fn) if (tp+fn)>0 else 0
    })
    print(f"  - Umbral {umbral:.2f}: Costo={costo}, FN={fn}, FP={fp}")

# Encontrar umbral óptimo
idx_optimo = np.argmin(costos)
umbral_optimo = umbrales[idx_optimo]
costo_minimo = costos[idx_optimo]

print(f"\n🎯 UMBRAL ÓPTIMO: {umbral_optimo:.2f}")
print(f"💰 COSTO MÍNIMO: {costo_minimo}")

# Gráfica de costo vs umbral
plt.figure(figsize=(10, 6))
plt.plot(umbrales, costos, marker='o', linewidth=2, color='blue', markersize=8)
plt.axvline(umbral_optimo, color='red', linestyle='--',
            label=f'Umbral óptimo = {umbral_optimo:.2f}', linewidth=2)
plt.xlabel('Umbral de decisión', fontsize=12)
plt.ylabel('Costo total', fontsize=12)
plt.title('Optimización de Umbral según Matriz de Costos', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 📊 Evaluación Final del Modelo

### Matriz de Confusión con Umbral Óptimo

La matriz de confusión muestra:
- **Verdaderos Positivos (TP)**: Malos pagadores correctamente identificados ✅
- **Falsos Negativos (FN)**: Malos pagadores clasificados como buenos ❌ (costo = 5)
- **Verdaderos Negativos (TN)**: Buenos pagadores correctamente identificados ✅
- **Falsos Positivos (FP)**: Buenos pagadores clasificados como malos ❌ (costo = 1)

### Métricas de Rendimiento

- **Precisión**: De los clasificados como malos, ¿cuántos realmente lo son?
- **Recall (Sensibilidad)**: De los malos pagadores reales, ¿cuántos detectamos?
- **F1-Score**: Media armónica de precisión y recall

In [ ]:
# ============================================
# MATRIZ DE CONFUSIÓN CON UMBRAL ÓPTIMO
# ============================================

y_pred_opt = (best_proba >= umbral_optimo).astype(int)
cm = confusion_matrix(y_test, y_pred_opt)

# Matriz de confusión
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Bueno (0)', 'Malo (1)'],
            yticklabels=['Bueno (0)', 'Malo (1)'])
ax1.set_title(f'Matriz de Confusión (Umbral = {umbral_optimo:.2f})',
              fontsize=14, fontweight='bold')
ax1.set_xlabel('Predicción', fontsize=12)
ax1.set_ylabel('Real', fontsize=12)

# Gráfico de barras con métricas
tn, fp, fn, tp = cm.ravel()
metrics = {
    'Verdaderos\nNegativos': tn,
    'Falsos\nPositivos': fp,
    'Falsos\nNegativos': fn,
    'Verdaderos\nPositivos': tp
}

colors = ['#2ecc71', '#e74c3c', '#e74c3c', '#2ecc71']
bars = ax2.bar(metrics.keys(), metrics.values(), color=colors, edgecolor='black', linewidth=1.5)
ax2.set_title('Desglose de Predicciones', fontsize=14, fontweight='bold')
ax2.set_ylabel('Número de registros', fontsize=12)

for bar, value in zip(bars, metrics.values()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             str(value), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Reporte de clasificación
print("\n📊 Reporte de Clasificación (Umbral óptimo):")
print(classification_report(y_test, y_pred_opt, target_names=['Bueno', 'Malo']))

print(f"\n💰 Resumen de Costos:")
print(f"  - Falsos Negativos: {fn} × 5 = {fn*5}")
print(f"  - Falsos Positivos: {fp} × 1 = {fp}")
print(f"  - Costo Total: {costo_minimo}")

## 📊 Distribución de Probabilidades

### Análisis de Separación de Clases

Esta visualización muestra cómo el modelo separa a los buenos y malos pagadores:

- **Buenos pagadores (verde)**: Concentrados en probabilidades bajas (< 0.5)
- **Malos pagadores (rojo)**: Concentrados en probabilidades más altas
- **Umbral óptimo (lí